In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-groq
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 123.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 68.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 11.8 kB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 204.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━

In [5]:
import os
files = os.listdir("/content")
for f in files:
    if f.endswith(".pdf"):
        print(f)

UniversityPhysicsVolume3.pdf
University_Physics_Volume_2.pdf
University_Physics_Volume_1.pdf


In [6]:
import os
from google.colab import userdata


GROQ_API_KEY = userdata.get("GROQ_API_KEY")


print("API key set ")

API key set 


In [7]:
!pip install langchain-text-splitters

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

all_docs = []


pdf_files = [
    "/content/UniversityPhysicsVolume3.pdf",
    "/content/University_Physics_Volume_2.pdf",
    "/content/University_Physics_Volume_1.pdf"
]

for pdf_file in pdf_files:
    print(f"Loading: {pdf_file}")
    loader = PyPDFLoader(pdf_file)
    docs = loader.load()
    all_docs.extend(docs)

print(f"\nTotal pages loaded: {len(all_docs)}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")

Loading: /content/UniversityPhysicsVolume3.pdf
Loading: /content/University_Physics_Volume_2.pdf
Loading: /content/University_Physics_Volume_1.pdf

Total pages loaded: 2345
Total chunks created: 7779


In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Loading embedding model (this may take 1–2 mins on first run)...")


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Creating vector store...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./physics_db"
)

vectorstore.persist()

print(f"Done! Stored {len(chunks)} chunks into ChromaDB ")

Loading embedding model (this may take 1–2 mins on first run)...


/tmp/ipykernel_2855/3611343177.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating vector store...
Done! Stored 7779 chunks into ChromaDB ✅


/tmp/ipykernel_2855/3611343177.py:19: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [12]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0.3,
    groq_api_key=GROQ_API_KEY
)


PROMPT_TEMPLATE = """
You are a helpful physics tutor. Use ONLY the context below to answer.

Rules:
- Explain the concept clearly and simply
- If the topic has a formula or equation, write it in simple plain text format
- Write equations like this examples:
    F = ma
    KE = (1/2) x m x v²
    F12 = -F21
    E = mc²
    v = u + at
- DO NOT use LaTeX code like \mathbf or \frac or [ ] or $$ symbols
- DO NOT use any brackets like [ ] around equations
- Just write the equation in clean simple text that anyone can read
- If the answer is not in the context, say: "I don't have information about that in my sources."

Context:
{context}

Question: {question}

Answer:
"""


prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
print("LLM and prompt ready ")


LLM and prompt ready 


<>:24: SyntaxWarning: invalid escape sequence '\m'
<>:24: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_2855/2803102355.py:24: SyntaxWarning: invalid escape sequence '\m'
  - DO NOT use LaTeX code like \mathbf or \frac or [ ] or $$ symbols


In [13]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built ")

RAG chain built 


In [14]:
from IPython.display import display, Markdown, HTML, clear_output


display(HTML("""
    <script>
        MathJax = {
            tex: { inlineMath: [['$', '$']], displayMath: [['$$', '$$']] }
        };
    </script>
    <script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js"></script>
"""))

def ask_physics_bot(question: str):
    answer = rag_chain.invoke(question)
    display(Markdown(f"**You asked:** {question}\n\n---\n\n### 🤖 Answer\n\n{answer}"))

In [15]:
ask_physics_bot("What is Newton's Third Law?")

**You asked:** What is Newton's Third Law?

---

### 🤖 Answer

Newton’s Third Law says that forces always come in pairs.  
If body A exerts a force on body B, then body B exerts an equal‑magnitude force on body A, but in the opposite direction.  
So the two forces are equal and opposite, but they act on different bodies, not on the same one.  
Because they act on different bodies, they do not cancel each other out.

In [17]:


ask_physics_bot("Explain the law of conservation of energy")

**You asked:** Explain the law of conservation of energy

---

### 🤖 Answer

The law of conservation of energy says that in an isolated system the total amount of energy never changes.  
That means if you add up all the different kinds of energy—kinetic, potential, thermal, chemical, etc.—the sum stays the same no matter what happens inside the system.  

The law is a very useful bookkeeping rule: it lets us track how energy moves from one form to another, but it is not something that can be derived from more basic laws.  No exception to the rule has ever been found.  

In everyday life the idea of “energy conservation” is a related philosophy that encourages people to use less energy by turning down thermostats, driving fewer miles, or making machines more efficient.  That philosophy is about reducing the amount of energy we use, while the conservation law is about the total energy staying constant in a closed system.

In [18]:
ask_physics_bot("What is Ohm's Law?")

**You asked:** What is Ohm's Law?

---

### 🤖 Answer

Ohm’s Law says that the electric current that flows through most ordinary materials is directly proportional to the voltage applied across them.  
In other words, if you double the voltage, the current also doubles (as long as the material behaves ohmically).

The relationship is written as  

V = I R  

or equivalently  

I = V / R  

where  
V = voltage (volts)  
I = current (amperes)  
R = resistance (ohms)

If a device follows this linear relationship, it is called an ohmic material or component. If it does not, it is called non‑ohmic.

In [19]:
ask_physics_bot("Explain Newton's Second Law with formula")

**You asked:** Explain Newton's Second Law with formula

---

### 🤖 Answer

**Newton’s Second Law**

Newton’s second law tells us how a force changes the motion of an object.  
It says that the net force acting on an object is equal to the mass of the object times its acceleration.

**Equation**

F = m a

or, solving for acceleration

a = F / m

**What it means**

* **F** is the total (net) force on the object.  
* **m** is the mass of the object.  
* **a** is the acceleration, the rate at which the object’s velocity changes.  

Because the equation is a vector equation, the direction of the force and the direction of the acceleration are the same.  

**Key points**

* If you apply a larger force, the acceleration becomes larger.  
* If the mass is larger, the same force produces a smaller acceleration.  
* The law is used to calculate how objects move when forces act on them.  

That is the core idea of Newton’s second law of motion.

In [20]:
ask_physics_bot("What is the equation for kinetic energy?")

**You asked:** What is the equation for kinetic energy?

---

### 🤖 Answer

The kinetic energy (KE) of a single particle is given by  

KE = (1/2) × m × v²  

where  
m = mass of the particle  
v = speed of the particle.  

This formula shows that kinetic energy is one‑half the product of the particle’s mass and the square of its speed.

In [16]:
print("Physics Bot is ready! Type 'quit' to stop.\n")

while True:
    question = input("You: ")
    if question.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    if question.strip() == "":
        continue
    ask_physics_bot(question)
    print()

Physics Bot is ready! Type 'quit' to stop.

You: exit
Goodbye!
